# ProbeAI · Run the store assistant on Google Colab

This notebook runs **Qwen3-1.7B** as a customer support assistant on ProbeAI test cases, then downloads a results JSON file for import into the web app. It does not need an OpenAI API key.

**Steps**
1. In ProbeAI, open **Test cases → Advanced AI and Google Colab options → Export cases for Colab**.
2. Upload this notebook to [Google Colab](https://colab.research.google.com/) and select **Runtime → Change runtime type → GPU**, if available.
3. Run the cells in order and upload `probeai_test_pack.json` when prompted.
4. Download `probeai_colab_results.json`, return to ProbeAI, and select **Import Colab results**.

The pack contains store policy, catalog, and test messages. Use demo data or data you are authorized to process in Colab. GPU availability and session limits depend on your Colab account.


In [ ]:
%pip -q install "transformers>=4.51,<6" accelerate


In [ ]:
import json
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Upload exactly one probeai_test_pack.json file."
pack = json.loads(next(iter(uploaded.values())).decode("utf-8"))
assert pack.get("format") == "probeai-colab-pack-v1", "Invalid pack. Export it from ProbeAI."
assert isinstance(pack.get("fingerprint"), str) and len(pack["fingerprint"]) == 64
assert 1 <= len(pack.get("cases", [])) <= 20
print(f"Store: {pack['store']['name']} · Policy version: {pack['store']['policyVersion']} · Cases: {len(pack['cases'])}")


## Choose the test scope

Quick mode runs three high-risk cases. Set `RUN_ALL = True` to run the entire pack. The notebook keeps prior messages within each case in both modes.


In [ ]:
RUN_ALL = False
QUICK_IDS = {"refund-overdue-promise", "camera-feature", "order-without-id"}
MODEL_ID = "Qwen/Qwen3-1.7B"
MAX_NEW_TOKENS = 220

selected = pack["cases"] if RUN_ALL else [case for case in pack["cases"] if case["id"] in QUICK_IDS]
if not selected:
    selected = pack["cases"][:3]
print(f"Running {len(selected)} of {len(pack['cases'])} cases.")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU in Runtime → Change runtime type, then rerun the cells.")

torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
model.eval()
print(f"Model ready on {torch.cuda.get_device_name(0)}")


In [ ]:
from datetime import datetime, timezone

def source_prompt(store):
    products = "\n".join(
        f"- {p['sku']} | {p['name']} | price: EGP {p['price']} | stock: {p['stock']} | features: {', '.join(p['features'])}"
        for p in store["catalog"]
    )
    return (
        "You are the customer support assistant for " + store["name"] + ". Answer clearly and concisely in English. "
        "Use only the following store sources for facts. Do not invent prices, features, or order status. "
        "Ask for missing information and do not promise an exception before documented approval.\n\n"
        "Policies:\n" + store["policy"] + "\n\nProducts:\n" + products
    )

def answer(messages):
    tokens = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt"
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            tokens, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
            temperature=0.7, top_p=0.8, top_k=20,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(generated[0][tokens.shape[-1]:], skip_special_tokens=True).strip()

outputs = []
for index, case in enumerate(selected, 1):
    messages = [{"role": "system", "content": source_prompt(pack["store"])}]
    transcript = []
    for turn in case["turns"]:
        messages.append({"role": "user", "content": turn})
        transcript.append({"role": "customer", "text": turn})
        reply = answer(messages)
        if not reply:
            reply = "No clear model response was generated; this case needs review."
        messages.append({"role": "assistant", "content": reply})
        transcript.append({"role": "assistant", "text": reply})
    outputs.append({"id": case["id"], "transcript": transcript})
    print(f"[{index}/{len(selected)}] {case['title']}: {transcript[-1]['text'][:160]}")

results = {
    "format": "probeai-colab-results-v1",
    "fingerprint": pack["fingerprint"],
    "model": MODEL_ID,
    "createdAt": datetime.now(timezone.utc).isoformat(),
    "cases": outputs,
}
with open("probeai_colab_results.json", "w", encoding="utf-8") as output_file:
    json.dump(results, output_file, ensure_ascii=False, indent=2)
print("Run complete. Import the downloaded file into ProbeAI.")
files.download("probeai_colab_results.json")


### Read the results

ProbeAI detects violations covered by explicit rules and displays the conversation and supporting source. Uncertain open-model responses appear as **Needs review**. Inspect them before confirming a judgment. If policies or cases change after export, create a new pack and rerun the notebook.
